# Regulatory Reporting & Core Banking Data Warehouse

### Databricks | PySpark | Delta Lake | Spark SQL

## Project Overview

This project demonstrates an end-to-end data engineering pipeline for processing core banking transactions using the Medallion Architecture (Bronze, Silver, Gold).

The pipeline performs:

- Raw transaction ingestion
- Data quality validation
- Quarantine handling
- AML risk classification
- Suspicious transaction detection
- Incremental data loading
- MERGE (UPSERT)
- Regulatory reporting

# Project Architecture

This project follows the Medallion Architecture:

- **Core Banking System** → Source of raw transaction data
- **Bronze Layer** → Stores raw Delta tables
- **Silver Layer** → Performs validation, cleansing, AML enrichment, and fraud detection
- **Quarantine Table** → Stores invalid transactions
- **Gold Layer** → Generates regulatory reports, customer summaries, and data quality metrics
- **Dashboards** → Provides business and compliance reporting

## Technologies Used

- Databricks
- Apache Spark
- PySpark
- Spark SQL
- Delta Lake
- Python

In [0]:
from pyspark.sql.functions import *

# Mock Core Banking Transactions
raw_data = [
    ("T1001", "ACCN-991", 4500.00, "Credit", "2026-08-01 09:30:00", "US"),
    ("T1002", "ACCN-442", -250.00, "Debit", "2026-08-01 10:15:00", "GB"),
    ("T1003", "ACCN-991", 1250000.00, "Credit", "2026-08-01 11:02:00", "KY"),   # High AML Risk
    ("T1004", "ACCN-105", None, "Credit", "2026-08-01 11:45:00", "US"),          # Null Amount
    ("T1005", "ACCN-202", -50000.00, "Debit", "2026-08-01 12:20:00", "CH"),      # High AML Risk
    ("T1006", "", 1200.00, "Credit", "2026-08-01 13:05:00", "US")                # Missing Account ID
]

columns = [
    "transaction_id",
    "account_id",
    "amount",
    "transaction_type",
    "timestamp",
    "counterparty_country"
]

df_ingest = spark.createDataFrame(raw_data, columns)

In [0]:
df_ingest.show(truncate=False)

+--------------+----------+---------+----------------+-------------------+--------------------+
|transaction_id|account_id|amount   |transaction_type|timestamp          |counterparty_country|
+--------------+----------+---------+----------------+-------------------+--------------------+
|T1001         |ACCN-991  |4500.0   |Credit          |2026-08-01 09:30:00|US                  |
|T1002         |ACCN-442  |-250.0   |Debit           |2026-08-01 10:15:00|GB                  |
|T1003         |ACCN-991  |1250000.0|Credit          |2026-08-01 11:02:00|KY                  |
|T1004         |ACCN-105  |NULL     |Credit          |2026-08-01 11:45:00|US                  |
|T1005         |ACCN-202  |-50000.0 |Debit           |2026-08-01 12:20:00|CH                  |
|T1006         |          |1200.0   |Credit          |2026-08-01 13:05:00|US                  |
+--------------+----------+---------+----------------+-------------------+--------------------+



In [0]:
df_ingest.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- counterparty_country: string (nullable = true)



# Bronze Layer

The Bronze layer stores raw banking transactions exactly as received from the source system.

Tasks performed:

- Raw data ingestion
- Delta table creation

In [0]:
df_ingest.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_core_banking_transactions")

In [0]:
spark.read.table("bronze_core_banking_transactions").show(truncate=False)

+--------------+----------+---------+----------------+-------------------+--------------------+
|transaction_id|account_id|amount   |transaction_type|timestamp          |counterparty_country|
+--------------+----------+---------+----------------+-------------------+--------------------+
|T1001         |ACCN-991  |4500.0   |Credit          |2026-08-01 09:30:00|US                  |
|T1002         |ACCN-442  |-250.0   |Debit           |2026-08-01 10:15:00|GB                  |
|T1003         |ACCN-991  |1250000.0|Credit          |2026-08-01 11:02:00|KY                  |
|T1004         |ACCN-105  |NULL     |Credit          |2026-08-01 11:45:00|US                  |
|T1005         |ACCN-202  |-50000.0 |Debit           |2026-08-01 12:20:00|CH                  |
|T1006         |          |1200.0   |Credit          |2026-08-01 13:05:00|US                  |
+--------------+----------+---------+----------------+-------------------+--------------------+



In [0]:
from pyspark.sql.functions import col, when, current_timestamp

bronze_df = spark.read.table("bronze_core_banking_transactions")

bronze_df.show()

+--------------+----------+---------+----------------+-------------------+--------------------+
|transaction_id|account_id|   amount|transaction_type|          timestamp|counterparty_country|
+--------------+----------+---------+----------------+-------------------+--------------------+
|         T1001|  ACCN-991|   4500.0|          Credit|2026-08-01 09:30:00|                  US|
|         T1002|  ACCN-442|   -250.0|           Debit|2026-08-01 10:15:00|                  GB|
|         T1003|  ACCN-991|1250000.0|          Credit|2026-08-01 11:02:00|                  KY|
|         T1004|  ACCN-105|     NULL|          Credit|2026-08-01 11:45:00|                  US|
|         T1005|  ACCN-202| -50000.0|           Debit|2026-08-01 12:20:00|                  CH|
|         T1006|          |   1200.0|          Credit|2026-08-01 13:05:00|                  US|
+--------------+----------+---------+----------------+-------------------+--------------------+



In [0]:
valid_data_condition = (
    col("amount").isNotNull() &
    (col("account_id") != "")
)

In [0]:
quarantine_df = bronze_df.filter(~valid_data_condition)

quarantine_df.show(truncate=False)

+--------------+----------+------+----------------+-------------------+--------------------+
|transaction_id|account_id|amount|transaction_type|timestamp          |counterparty_country|
+--------------+----------+------+----------------+-------------------+--------------------+
|T1004         |ACCN-105  |NULL  |Credit          |2026-08-01 11:45:00|US                  |
|T1006         |          |1200.0|Credit          |2026-08-01 13:05:00|US                  |
+--------------+----------+------+----------------+-------------------+--------------------+



In [0]:
quarantine_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("quarantine_failed_transactions")

In [0]:
spark.read.table("quarantine_failed_transactions").show(truncate=False)

+--------------+----------+------+----------------+-------------------+--------------------+
|transaction_id|account_id|amount|transaction_type|timestamp          |counterparty_country|
+--------------+----------+------+----------------+-------------------+--------------------+
|T1004         |ACCN-105  |NULL  |Credit          |2026-08-01 11:45:00|US                  |
|T1006         |          |1200.0|Credit          |2026-08-01 13:05:00|US                  |
+--------------+----------+------+----------------+-------------------+--------------------+



# Silver Layer

The Silver layer cleanses and enriches banking transactions.

Tasks performed:

- Data validation
- Remove invalid records
- Create quarantine table
- AML Risk Classification
- Large Transaction Detection
- Suspicious Transaction Detection

In [0]:
silver_df = (
    bronze_df
    .filter(valid_data_condition)
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
    .withColumn(
        "aml_risk_tier",
        when(col("counterparty_country").isin("KY", "CH"), "HIGH")
        .otherwise("LOW")
    )
    .withColumn("ingested_at", current_timestamp())
)

In [0]:
silver_df.show(truncate=False)

+--------------+----------+---------+----------------+-------------------+--------------------+-------------+--------------------------+
|transaction_id|account_id|amount   |transaction_type|timestamp          |counterparty_country|aml_risk_tier|ingested_at               |
+--------------+----------+---------+----------------+-------------------+--------------------+-------------+--------------------------+
|T1001         |ACCN-991  |4500.0   |Credit          |2026-08-01 09:30:00|US                  |LOW          |2026-08-03 16:49:13.345382|
|T1002         |ACCN-442  |-250.0   |Debit           |2026-08-01 10:15:00|GB                  |LOW          |2026-08-03 16:49:13.345382|
|T1003         |ACCN-991  |1250000.0|Credit          |2026-08-01 11:02:00|KY                  |HIGH         |2026-08-03 16:49:13.345382|
|T1005         |ACCN-202  |-50000.0 |Debit           |2026-08-01 12:20:00|CH                  |HIGH         |2026-08-03 16:49:13.345382|
+--------------+----------+---------+----

In [0]:
silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_transactions")

In [0]:
spark.read.table("silver_transactions").show(truncate=False)

+--------------+----------+---------+----------------+-------------------+--------------------+-------------+--------------------------+----------------------+---------------+
|transaction_id|account_id|amount   |transaction_type|timestamp          |counterparty_country|aml_risk_tier|ingested_at               |large_transaction_flag|suspicious_flag|
+--------------+----------+---------+----------------+-------------------+--------------------+-------------+--------------------------+----------------------+---------------+
|T1001         |ACCN-991  |4500.0   |Credit          |2026-08-01 09:30:00|US                  |LOW          |2026-08-03 16:50:20.587568|NULL                  |NULL           |
|T1002         |ACCN-442  |-250.0   |Debit           |2026-08-01 10:15:00|GB                  |LOW          |2026-08-03 16:50:20.587568|NULL                  |NULL           |
|T1003         |ACCN-991  |1250000.0|Credit          |2026-08-01 11:02:00|KY                  |HIGH         |2026-08-03 

In [0]:
%sql
CREATE OR REPLACE TABLE gold_regulatory_reporting_daily AS

SELECT
    DATE(timestamp) AS report_date,
    aml_risk_tier,

    COUNT(transaction_id) AS total_transaction_count,

    SUM(
        CASE
            WHEN amount > 0 THEN amount
            ELSE 0
        END
    ) AS total_inflow_volume,

    SUM(
        CASE
            WHEN amount < 0 THEN ABS(amount)
            ELSE 0
        END
    ) AS total_outflow_volume,

    SUM(amount) AS net_liquidity_impact

FROM silver_transactions

GROUP BY
DATE(timestamp),
aml_risk_tier;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM gold_regulatory_reporting_daily;

report_date,aml_risk_tier,total_transaction_count,total_inflow_volume,total_outflow_volume,net_liquidity_impact
2026-08-01,LOW,2,4500.0,250.0,4250.0
2026-08-01,HIGH,2,1250000.0,50000.0,1200000.0


In [0]:
from pyspark.sql.functions import abs, when, col

silver_df = spark.read.table("silver_transactions")

silver_df = silver_df.withColumn(
    "large_transaction_flag",
    when(abs(col("amount")) >= 1000000, "YES")
    .otherwise("NO")
)

silver_df.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_transactions")

In [0]:
spark.read.table("silver_transactions").select(
    "transaction_id",
    "amount",
    "large_transaction_flag"
).show()

+--------------+---------+----------------------+
|transaction_id|   amount|large_transaction_flag|
+--------------+---------+----------------------+
|         T1001|   4500.0|                    NO|
|         T1002|   -250.0|                    NO|
|         T1003|1250000.0|                   YES|
|         T1005| -50000.0|                    NO|
+--------------+---------+----------------------+



In [0]:
from pyspark.sql.functions import abs, when, col

silver_df = spark.read.table("silver_transactions")

silver_df = silver_df.withColumn(
    "suspicious_flag",
    when(
        (col("aml_risk_tier") == "HIGH") &
        (abs(col("amount")) >= 100000),
        "YES"
    ).otherwise("NO")
)

silver_df.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_transactions")

In [0]:
spark.read.table("silver_transactions").select(
    "transaction_id",
    "aml_risk_tier",
    "amount",
    "suspicious_flag"
).show()

+--------------+-------------+---------+---------------+
|transaction_id|aml_risk_tier|   amount|suspicious_flag|
+--------------+-------------+---------+---------------+
|         T1001|          LOW|   4500.0|             NO|
|         T1002|          LOW|   -250.0|             NO|
|         T1003|         HIGH|1250000.0|            YES|
|         T1005|         HIGH| -50000.0|             NO|
+--------------+-------------+---------+---------------+



In [0]:
%sql

CREATE OR REPLACE TABLE gold_customer_summary AS

SELECT
    account_id,
    COUNT(*) AS total_transactions,
    SUM(amount) AS net_amount,
    MAX(amount) AS highest_transaction,
    MIN(amount) AS lowest_transaction

FROM silver_transactions

GROUP BY account_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_customer_summary;

account_id,total_transactions,net_amount,highest_transaction,lowest_transaction
ACCN-442,1,-250.0,-250.0,-250.0
ACCN-991,2,1254500.0,1250000.0,4500.0
ACCN-202,1,-50000.0,-50000.0,-50000.0


In [0]:
spark.read.table("silver_transactions").select(
    "transaction_id",
    "amount",
    "large_transaction_flag"
).show()

+--------------+---------+----------------------+
|transaction_id|   amount|large_transaction_flag|
+--------------+---------+----------------------+
|         T1001|   4500.0|                    NO|
|         T1002|   -250.0|                    NO|
|         T1003|1250000.0|                   YES|
|         T1005| -50000.0|                    NO|
+--------------+---------+----------------------+



In [0]:
from pyspark.sql.functions import abs, when, col

silver_df = spark.read.table("silver_transactions")

silver_df = silver_df.withColumn(
    "suspicious_flag",
    when(
        (col("aml_risk_tier") == "HIGH") &
        (abs(col("amount")) >= 100000),
        "YES"
    ).otherwise("NO")
)

silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_transactions")

In [0]:
spark.read.table("silver_transactions").select(
    "transaction_id",
    "aml_risk_tier",
    "amount",
    "suspicious_flag"
).show()

+--------------+-------------+---------+---------------+
|transaction_id|aml_risk_tier|   amount|suspicious_flag|
+--------------+-------------+---------+---------------+
|         T1001|          LOW|   4500.0|             NO|
|         T1002|          LOW|   -250.0|             NO|
|         T1003|         HIGH|1250000.0|            YES|
|         T1005|         HIGH| -50000.0|             NO|
+--------------+-------------+---------+---------------+



# Gold Layer

The Gold layer contains business-ready datasets used for reporting and analytics.

Tables created:

- Regulatory Reporting
- Customer Summary
- Data Quality Metrics

In [0]:
%sql

CREATE OR REPLACE TABLE gold_customer_summary AS

SELECT
    account_id,
    COUNT(*) AS total_transactions,
    SUM(amount) AS net_amount,
    MAX(amount) AS highest_transaction,
    MIN(amount) AS lowest_transaction
FROM silver_transactions
GROUP BY account_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_customer_summary
ORDER BY account_id;

account_id,total_transactions,net_amount,highest_transaction,lowest_transaction
ACCN-202,1,-50000.0,-50000.0,-50000.0
ACCN-442,1,-250.0,-250.0,-250.0
ACCN-991,2,1254500.0,1250000.0,4500.0


In [0]:
from pyspark.sql import Row

bronze_count = spark.read.table("bronze_core_banking_transactions").count()
silver_count = spark.read.table("silver_transactions").count()
quarantine_count = spark.read.table("quarantine_failed_transactions").count()

metrics = [
    Row(metric="Bronze Records", value=bronze_count),
    Row(metric="Silver Records", value=silver_count),
    Row(metric="Rejected Records", value=quarantine_count)
]

metrics_df = spark.createDataFrame(metrics)

metrics_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_data_quality_metrics")

In [0]:
spark.read.table("gold_data_quality_metrics").show()

+----------------+-----+
|          metric|value|
+----------------+-----+
|  Bronze Records|    6|
|  Silver Records|    4|
|Rejected Records|    2|
+----------------+-----+



In [0]:
new_transactions = [
    ("T1007", "ACCN-555", 8000.0, "Credit", "2026-08-02 09:00:00", "US"),
    ("T1008", "ACCN-991", -1200.0, "Debit", "2026-08-02 09:15:00", "GB"),
    ("T1009", "ACCN-777", 2500000.0, "Credit", "2026-08-02 10:00:00", "KY")
]

new_df = spark.createDataFrame(new_transactions, columns)

new_df.show()

+--------------+----------+---------+----------------+-------------------+--------------------+
|transaction_id|account_id|   amount|transaction_type|          timestamp|counterparty_country|
+--------------+----------+---------+----------------+-------------------+--------------------+
|         T1007|  ACCN-555|   8000.0|          Credit|2026-08-02 09:00:00|                  US|
|         T1008|  ACCN-991|  -1200.0|           Debit|2026-08-02 09:15:00|                  GB|
|         T1009|  ACCN-777|2500000.0|          Credit|2026-08-02 10:00:00|                  KY|
+--------------+----------+---------+----------------+-------------------+--------------------+



In [0]:
new_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("bronze_core_banking_transactions")

In [0]:
spark.read.table("bronze_core_banking_transactions").count()

9

# Incremental Data Loading

New banking transactions are appended to the Bronze layer.

Only new records are processed and merged into the Silver layer.

In [0]:
incremental_df = spark.read.table("bronze_core_banking_transactions") \
    .filter(col("transaction_id").isin("T1007", "T1008", "T1009"))

incremental_df.show()

+--------------+----------+---------+----------------+-------------------+--------------------+
|transaction_id|account_id|   amount|transaction_type|          timestamp|counterparty_country|
+--------------+----------+---------+----------------+-------------------+--------------------+
|         T1007|  ACCN-555|   8000.0|          Credit|2026-08-02 09:00:00|                  US|
|         T1008|  ACCN-991|  -1200.0|           Debit|2026-08-02 09:15:00|                  GB|
|         T1009|  ACCN-777|2500000.0|          Credit|2026-08-02 10:00:00|                  KY|
+--------------+----------+---------+----------------+-------------------+--------------------+



In [0]:
incremental_df = (
    incremental_df
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
    .withColumn(
        "aml_risk_tier",
        when(col("counterparty_country").isin("KY", "CH"), "HIGH")
        .otherwise("LOW")
    )
    .withColumn(
        "large_transaction_flag",
        when(abs(col("amount")) >= 1000000, "YES").otherwise("NO")
    )
    .withColumn(
        "suspicious_flag",
        when(
            (col("counterparty_country").isin("KY", "CH")) &
            (abs(col("amount")) >= 100000),
            "YES"
        ).otherwise("NO")
    )
    .withColumn("ingested_at", current_timestamp())
)

In [0]:
incremental_df.createOrReplaceTempView("incremental_transactions")

# MERGE (UPSERT)

Delta Lake MERGE updates existing records and inserts new transactions.

This avoids rebuilding the entire Silver table.

In [0]:
%sql

MERGE INTO silver_transactions AS target

USING incremental_transactions AS source

ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
spark.read.table("silver_transactions").count()

7

In [0]:
spark.read.table("silver_transactions") \
.select(
    "transaction_id",
    "account_id",
    "amount",
    "aml_risk_tier",
    "large_transaction_flag",
    "suspicious_flag"
).show(truncate=False)

+--------------+----------+---------+-------------+----------------------+---------------+
|transaction_id|account_id|amount   |aml_risk_tier|large_transaction_flag|suspicious_flag|
+--------------+----------+---------+-------------+----------------------+---------------+
|T1001         |ACCN-991  |4500.0   |LOW          |NO                    |NO             |
|T1002         |ACCN-442  |-250.0   |LOW          |NO                    |NO             |
|T1003         |ACCN-991  |1250000.0|HIGH         |YES                   |YES            |
|T1005         |ACCN-202  |-50000.0 |HIGH         |NO                    |NO             |
|T1007         |ACCN-555  |8000.0   |LOW          |NO                    |NO             |
|T1008         |ACCN-991  |-1200.0  |LOW          |NO                    |NO             |
|T1009         |ACCN-777  |2500000.0|HIGH         |YES                   |YES            |
+--------------+----------+---------+-------------+----------------------+---------------+

In [0]:
%sql
DESCRIBE HISTORY silver_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
11,2026-08-03T17:01:53.000Z,73361283820944,bachinalalith87@gmail.com,MERGE,"Map(predicate -> [""(transaction_id#24588 = transaction_id#24556)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2495189163994649),cc03599b-a45d-4658-8050-835179bb409b,0803-151526-zf4car8k-v2n,10,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3125, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1796, materializeSourceTimeMs -> 272, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 548, numTargetRowsUpdated -> 0, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 939)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
10,2026-08-03T16:57:46.000Z,73361283820944,bachinalalith87@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2495189163994649),d089fa27-2e70-4d80-985c-303d6a8cc27c,0803-151526-zf4car8k-v2n,9,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3197, numDeletionVectorsRemoved -> 0, numOutputRows -> 4, numOutputBytes -> 3197)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
9,2026-08-03T16:53:03.000Z,73361283820944,bachinalalith87@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2495189163994649),17de1f55-62b4-4c4f-80e9-eb6b0440894e,0803-151526-zf4car8k-v2n,8,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3154, numDeletionVectorsRemoved -> 0, numOutputRows -> 4, numOutputBytes -> 3197)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
8,2026-08-03T16:52:26.000Z,73361283820944,bachinalalith87@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2495189163994649),f54669fb-875e-4bc6-a1c8-3a3c9bd3da57,0803-151526-zf4car8k-v2n,7,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2690, numDeletionVectorsRemoved -> 0, numOutputRows -> 4, numOutputBytes -> 3154)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
7,2026-08-03T16:50:22.000Z,73361283820944,bachinalalith87@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.pa

In [0]:
%sql
SELECT *
FROM silver_transactions VERSION AS OF 5;

transaction_id,account_id,amount,transaction_type,timestamp,counterparty_country,aml_risk_tier,ingested_at,large_transaction_flag,suspicious_flag
T1001,ACCN-991,4500.0,Credit,2026-08-01T09:30:00.000Z,US,LOW,2026-08-03T15:43:58.271Z,NO,NO
T1002,ACCN-442,-250.0,Debit,2026-08-01T10:15:00.000Z,GB,LOW,2026-08-03T15:43:58.271Z,NO,NO
T1003,ACCN-991,1250000.0,Credit,2026-08-01T11:02:00.000Z,KY,HIGH,2026-08-03T15:43:58.271Z,YES,YES
T1005,ACCN-202,-50000.0,Debit,2026-08-01T12:20:00.000Z,CH,HIGH,2026-08-03T15:43:58.271Z,NO,NO
T1007,ACCN-555,8000.0,Credit,2026-08-02T09:00:00.000Z,US,LOW,2026-08-03T16:04:55.961Z,NO,NO
T1008,ACCN-991,-1200.0,Debit,2026-08-02T09:15:00.000Z,GB,LOW,2026-08-03T16:04:55.961Z,NO,NO
T1009,ACCN-777,2500000.0,Credit,2026-08-02T10:00:00.000Z,KY,HIGH,2026-08-03T16:04:55.961Z,YES,YES


In [0]:
%sql

SELECT COUNT(*)
FROM silver_transactions;

COUNT(*)
7


In [0]:
%sql

OPTIMIZE silver_transactions;

path,metrics
,"List(1, 2, List(3383, 3383, 3383.0, 1, 3383), List(3125, 3197, 3161.0, 2, 6322), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1785776660166, 1785776661972, 8, 1, null, List(0, 0), null, 10, 10, 441, 0, null, null)"


In [0]:
%sql
SELECT *
FROM gold_regulatory_reporting_daily;

report_date,aml_risk_tier,total_transaction_count,total_inflow_volume,total_outflow_volume,net_liquidity_impact
2026-08-01,LOW,2,4500.0,250.0,4250.0
2026-08-01,HIGH,2,1250000.0,50000.0,1200000.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_regulatory_reporting_daily AS

SELECT
    DATE(timestamp) AS report_date,
    aml_risk_tier,
    COUNT(transaction_id) AS total_transaction_count,
    SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END) AS total_inflow_volume,
    SUM(CASE WHEN amount < 0 THEN ABS(amount) ELSE 0 END) AS total_outflow_volume,
    SUM(amount) AS net_liquidity_impact
FROM silver_transactions
GROUP BY DATE(timestamp), aml_risk_tier;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_regulatory_reporting_daily
ORDER BY report_date, aml_risk_tier;

report_date,aml_risk_tier,total_transaction_count,total_inflow_volume,total_outflow_volume,net_liquidity_impact
2026-08-01,HIGH,2,1250000.0,50000.0,1200000.0
2026-08-01,LOW,2,4500.0,250.0,4250.0
2026-08-02,HIGH,1,2500000.0,0.0,2500000.0
2026-08-02,LOW,2,8000.0,1200.0,6800.0


# Dashboard Queries

The following SQL queries are used to create dashboards for:

- AML Risk Distribution
- Daily Transaction Volume
- Customer Summary
- Large Transactions
- Suspicious Transactions

# Conclusion

This project demonstrates an enterprise-style banking data warehouse using Databricks and Delta Lake.

Key features include:

- Medallion Architecture
- Delta Tables
- Data Quality Validation
- AML Classification
- Incremental Processing
- MERGE (UPSERT)
- Regulatory Reporting